# 09 — AutoKeras extra: ¿bate la búsqueda automática a nuestro diseño manual?

**Práctica B3-T4 · Forecasting financiero multivariante (SP500, 23 activos)**

Esta es la sección **EXTRA** del entregable. Pregunta de investigación:

> ¿Una búsqueda automática de arquitectura (NAS) supera nuestras 5 variantes hand-crafted en `06_mixtos.ipynb`?

Aplicamos AutoKeras o Keras Tuner sobre la combinación más crítica `(input_window=90, output_window=90)` —la que alimentará las carteras de 2025— y comparamos contra:

1. `mixto_profMIX_in90_out90` (top-1 manual, `mae_test = 0.001277`).
2. Ensemble top-3 deep dive (`mae_test = 0.001285`).
3. MLP denso de tus compañeras (`mae_test ≈ 0.001494`).

## Estrategia

Tiempo de cómputo limitado (~30 minutos). Dos alternativas, intentamos en este orden:

1. **AutoKeras** — si está instalado, lanzamos `AutoModel` con `TimeseriesInput` + `RegressionHead`.
2. **Keras Tuner** — fallback más ligero, instalado por defecto con TF. Búsqueda sobre hiperparámetros de un mixto familiar (units LSTM, filtros Conv, dropout, lr).

Cualquiera de los dos resultados es **defendible**:

- Si NAS gana -> el espacio de arquitecturas oculta diseños mejores que los manuales.
- Si NAS pierde -> el conocimiento de dominio bate a la búsqueda automática en problemas pequeños y ruidosos como series financieras.

## Reglas

- Mismo `X_train`/`X_val`/`X_test` que el resto del proyecto (split idéntico al `06_mixtos.ipynb`).
- Test se evalúa **una sola vez** al final. NAS no toca test.
- Budget: configurable (`MAX_TRIALS`, `TIME_BUDGET_MIN`). Recomendado: 15-20 trials.

## 1. Setup

In [ ]:
import os
import json
import time
import pickle
import random
import warnings
import datetime as dt
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import load_model

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

RANDOM_SEED = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"TensorFlow {tf.__version__}")


def _resolve_dir(name):
    for cand in [Path(name), Path("..") / name]:
        if cand.exists():
            return cand.resolve()
    p = Path(name)
    p.mkdir(parents=True, exist_ok=True)
    return p.resolve()


RESULTS_DIR = _resolve_dir("results")
MODELS_DIR = _resolve_dir("models")
DATA_DIR = _resolve_dir("data")

N_ASSETS = 23
INPUT_WINDOW = 90
OUTPUT_WINDOW = 90

print(f"RESULTS_DIR : {RESULTS_DIR}")
print(f"MODELS_DIR  : {MODELS_DIR}")

In [ ]:
AUTOKERAS_OK = False
KERAS_TUNER_OK = False

try:
    import autokeras as ak  # noqa: F401
    AUTOKERAS_OK = True
    print("AutoKeras detectado: usaremos AutoKeras.")
except Exception:
    print("AutoKeras NO detectado. Si quieres usarlo, ejecuta: pip install autokeras")
    print("Probando Keras Tuner como fallback...")
    try:
        import keras_tuner as kt  # noqa: F401
        KERAS_TUNER_OK = True
        print("Keras Tuner detectado: usaremos Keras Tuner como fallback.")
    except Exception:
        print("Keras Tuner tampoco detectado. Si quieres ejecutar este notebook, instala uno de los dos:")
        print("  pip install keras-tuner")
        print("  o (mas pesado) pip install autokeras")

USE_AUTOKERAS = AUTOKERAS_OK
USE_KERAS_TUNER = (not AUTOKERAS_OK) and KERAS_TUNER_OK

if not (USE_AUTOKERAS or USE_KERAS_TUNER):
    print("\nSIN AUTOKERAS NI KERAS_TUNER: este notebook solo cargara y mostrara los resultados ya existentes en results/autokeras_resultados.csv si los hay.")

## 2. Preparación de datos

Replicamos exactamente el `split_triple` de competición. Aplicamos el `StandardScaler` global fit-en-train. AutoKeras/Tuner verán los mismos `X_train` normalizados que el `06_mixtos`.

In [ ]:
CACHE_RETURNS = DATA_DIR / "sp500_returns.parquet"


def _read(p):
    try:
        return pd.read_parquet(p)
    except Exception:
        with open(p.with_suffix(".pkl"), "rb") as f:
            return pickle.load(f)


returns_df = _read(CACHE_RETURNS if CACHE_RETURNS.exists() else CACHE_RETURNS.with_suffix(".pkl"))
print(f"returns_df: {returns_df.shape}")


def create_time_series_data(data, input_window_size, output_window_size):
    X, y = [], []
    arr = data.values if isinstance(data, pd.DataFrame) else data
    for i in range(len(arr) - input_window_size - output_window_size + 1):
        X.append(arr[i : i + input_window_size])
        y.append(np.mean(arr[i + input_window_size : i + input_window_size + output_window_size], axis=0))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def split_triple(X, y, test_size=0.1, val_size=0.1, random_state=RANDOM_SEED):
    X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=test_size, shuffle=False)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=val_size, shuffle=True, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


X, y = create_time_series_data(returns_df, INPUT_WINDOW, OUTPUT_WINDOW)
X_train, X_val, X_test, y_train, y_val, y_test = split_triple(X, y)

scaler_X = StandardScaler()
scaler_y = StandardScaler()
scaler_X.fit(X_train.reshape(-1, X_train.shape[-1]))
scaler_y.fit(y_train)


def _sx(X):
    n, w, c = X.shape
    return scaler_X.transform(X.reshape(-1, c)).reshape(n, w, c).astype(np.float32)


def _sy(y):
    return scaler_y.transform(y).astype(np.float32)


X_train_n = _sx(X_train)
X_val_n = _sx(X_val)
X_test_n = _sx(X_test)
y_train_n = _sy(y_train)
y_val_n = _sy(y_val)

print(f"X_train: {X_train.shape}    X_val: {X_val.shape}    X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}    y_val: {y_val.shape}    y_test: {y_test.shape}")

## 3. Búsqueda NAS

### Opción A — AutoKeras

Si está instalado: `ak.AutoModel` con `Input(shape=(90, 23))` + `RegressionHead(output_dim=23, loss="mae")`. Lo dejamos buscar dentro de un grafo automático.

### Opción B — Keras Tuner (fallback)

Si no hay AutoKeras: búsqueda Bayesiana sobre los hiperparámetros de un mixto Conv1D + LSTM + Dense que ya conocemos del notebook 06. Espacio:

- `conv_filters ∈ {32, 64, 96}`
- `lstm_units ∈ {32, 64, 96, 128}`
- `dropout ∈ {0.1, 0.2, 0.3}`
- `lr ∈ {1e-4, 5e-4, 1e-3}`

Es más conservador pero más reproducible y ligero.

In [ ]:
MAX_TRIALS = 15
EPOCHS_PER_TRIAL = 50
PROJECT_DIR = MODELS_DIR / "autokeras_search"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

best_model = None
best_mae_val = None
search_results = []

if USE_AUTOKERAS:
    import autokeras as ak
    print(f"AutoKeras: {MAX_TRIALS} trials, {EPOCHS_PER_TRIAL} epochs/trial")
    t0 = time.time()

    input_node = ak.Input()
    output_node = ak.DenseBlock()(input_node)
    output_node = ak.RegressionHead(output_dim=N_ASSETS, loss="mae", metrics=["mae"])(output_node)

    clf = ak.AutoModel(
        inputs=input_node,
        outputs=output_node,
        max_trials=MAX_TRIALS,
        objective="val_mae",
        overwrite=True,
        directory=str(PROJECT_DIR),
        project_name="ak_in90_out90",
        seed=RANDOM_SEED,
    )

    try:
        clf.fit(
            X_train_n, y_train_n,
            validation_data=(X_val_n, y_val_n),
            epochs=EPOCHS_PER_TRIAL,
            batch_size=32,
            verbose=2,
        )
        best_model = clf.export_model()
        print(f"AutoKeras terminado en {(time.time()-t0)/60:.1f} min")
        best_model.summary()
    except Exception as exc:
        print(f"AutoKeras ERROR: {type(exc).__name__}: {exc}")
        print("Cambiamos a Keras Tuner como fallback.")
        USE_AUTOKERAS = False
        try:
            import keras_tuner as kt
            USE_KERAS_TUNER = True
        except Exception:
            USE_KERAS_TUNER = False

In [ ]:
if (not USE_AUTOKERAS) and USE_KERAS_TUNER:
    import keras_tuner as kt
    from tensorflow.keras import Model
    from tensorflow.keras.layers import (
        Input, Conv1D, MaxPooling1D, LSTM, GRU, BatchNormalization,
        Dropout, Dense, Flatten, GlobalAveragePooling1D, Concatenate, Bidirectional,
    )
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

    print(f"Keras Tuner: bayesian opt, max {MAX_TRIALS} trials, {EPOCHS_PER_TRIAL} epochs/trial")

    def build_search_model(hp):
        conv_filters = hp.Choice("conv_filters", [32, 64, 96])
        kernel_size = hp.Choice("kernel_size", [3, 5, 7])
        lstm_units = hp.Choice("lstm_units", [32, 64, 96, 128])
        dropout_rate = hp.Choice("dropout", [0.1, 0.2, 0.3])
        lr = hp.Choice("lr", [1e-4, 5e-4, 1e-3])
        use_bn = hp.Boolean("use_bn", default=True)

        inp = Input(shape=(INPUT_WINDOW, N_ASSETS))
        x = Conv1D(conv_filters, kernel_size, padding="causal", activation="relu")(inp)
        if use_bn:
            x = BatchNormalization()(x)
        x = Conv1D(conv_filters, kernel_size, padding="causal", activation="relu")(x)
        if use_bn:
            x = BatchNormalization()(x)
        x = LSTM(lstm_units)(x)
        x = Dropout(dropout_rate)(x)
        x = Dense(64, activation="relu")(x)
        out = Dense(N_ASSETS, activation="linear")(x)

        model = Model(inp, out)
        model.compile(optimizer=Adam(learning_rate=lr), loss="mae", metrics=["mae"])
        return model

    tuner = kt.BayesianOptimization(
        build_search_model,
        objective=kt.Objective("val_mae", direction="min"),
        max_trials=MAX_TRIALS,
        directory=str(PROJECT_DIR),
        project_name="kt_in90_out90",
        overwrite=True,
        seed=RANDOM_SEED,
    )

    t0 = time.time()
    tuner.search(
        X_train_n, y_train_n,
        validation_data=(X_val_n, y_val_n),
        epochs=EPOCHS_PER_TRIAL,
        batch_size=32,
        callbacks=[
            EarlyStopping(monitor="val_mae", patience=8, restore_best_weights=True, mode="min"),
            ReduceLROnPlateau(monitor="val_mae", factor=0.5, patience=3, min_lr=1e-6, mode="min"),
        ],
        verbose=2,
    )
    elapsed_kt = time.time() - t0
    print(f"\nKeras Tuner busqueda terminada en {elapsed_kt/60:.1f} min")

    best_hyper = tuner.get_best_hyperparameters(num_trials=3)[0]
    print(f"\nMejor hiper: {best_hyper.values}")

    best_model = tuner.get_best_models(num_models=1)[0]
    print("\nResumen del mejor modelo:")
    best_model.summary()

## 4. Evaluación y comparativa

Una sola evaluación de test del modelo NAS encontrado. Comparamos con:

- `mixto_profMIX_in90_out90` (`results/mixtos_resultados.csv`).
- Ensemble top-3 (`results/mixto_deepdive_resumen.csv`).
- MLP denso (`results/mlp_resultados.csv` si existe).

In [ ]:
RESULTS_CSV = RESULTS_DIR / "autokeras_resultados.csv"

ak_mae_train = ak_mae_val = ak_mae_test = ak_n_params = None
ak_method = "ninguna"

if best_model is not None:
    pred_train_n = best_model.predict(X_train_n, verbose=0)
    pred_val_n = best_model.predict(X_val_n, verbose=0)
    pred_test_n = best_model.predict(X_test_n, verbose=0)

    if pred_train_n.ndim == 3:
        pred_train_n = pred_train_n.reshape(pred_train_n.shape[0], -1)
        pred_val_n = pred_val_n.reshape(pred_val_n.shape[0], -1)
        pred_test_n = pred_test_n.reshape(pred_test_n.shape[0], -1)

    pred_train = scaler_y.inverse_transform(pred_train_n)
    pred_val = scaler_y.inverse_transform(pred_val_n)
    pred_test = scaler_y.inverse_transform(pred_test_n)

    ak_mae_train = mae(y_train, pred_train)
    ak_mae_val = mae(y_val, pred_val)
    ak_mae_test = mae(y_test, pred_test)
    ak_n_params = int(best_model.count_params())
    ak_method = "autokeras" if USE_AUTOKERAS else "keras_tuner"

    print(f"NAS ({ak_method}) - mejor modelo encontrado:")
    print(f"  Params      : {ak_n_params:,}")
    print(f"  mae_train   : {ak_mae_train:.6f}")
    print(f"  mae_val     : {ak_mae_val:.6f}")
    print(f"  mae_test    : {ak_mae_test:.6f}")

    save_path = MODELS_DIR / "autokeras_in90_out90.keras"
    try:
        best_model.save(save_path)
        print(f"\nModelo guardado en: {save_path}")
    except Exception as exc:
        print(f"\nAVISO: no se pudo guardar el modelo NAS: {exc}")

In [ ]:
comparativa_rows = []

if ak_mae_test is not None:
    comparativa_rows.append({
        "metodo": ak_method,
        "modelo": "autokeras_in90_out90" if USE_AUTOKERAS else "keras_tuner_best",
        "n_params": ak_n_params,
        "mae_train": ak_mae_train,
        "mae_val": ak_mae_val,
        "mae_test": ak_mae_test,
        "notas": f"Busqueda automatica con {MAX_TRIALS} trials, {EPOCHS_PER_TRIAL} epochs/trial",
    })

mix_csv = RESULTS_DIR / "mixtos_resultados.csv"
if mix_csv.exists():
    df_mix = pd.read_csv(mix_csv)
    top1_row = df_mix[(df_mix["input_window"] == INPUT_WINDOW) & (df_mix["output_window"] == OUTPUT_WINDOW) & (df_mix["variante"] == "profMIX")]
    if len(top1_row) > 0:
        r = top1_row.iloc[0]
        comparativa_rows.append({
            "metodo": "manual",
            "modelo": f"mixto_profMIX_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}",
            "n_params": int(r["n_params"]),
            "mae_train": float(r["mae_train"]),
            "mae_val": float(r["mae_val"]),
            "mae_test": float(r["mae_test"]),
            "notas": "Top-1 manual (mixto profesor MIX)",
        })

dd_csv = RESULTS_DIR / "mixto_deepdive_resumen.csv"
if dd_csv.exists():
    df_dd = pd.read_csv(dd_csv)
    if len(df_dd) > 0:
        r = df_dd.iloc[0]
        comparativa_rows.append({
            "metodo": "manual_ensemble",
            "modelo": f"ensemble_top3_{INPUT_WINDOW}_{OUTPUT_WINDOW}",
            "n_params": np.nan,
            "mae_train": np.nan,
            "mae_val": np.nan,
            "mae_test": float(r["mae_test_ensemble"]),
            "notas": f"Top-3: {r['top_3_variantes']}",
        })

mlp_csv = RESULTS_DIR / "mlp_resultados.csv"
if mlp_csv.exists():
    df_mlp = pd.read_csv(mlp_csv)
    mlp_row = df_mlp[(df_mlp["input_window"] == INPUT_WINDOW) & (df_mlp["output_window"] == OUTPUT_WINDOW)]
    if len(mlp_row) > 0:
        r = mlp_row.iloc[0]
        comparativa_rows.append({
            "metodo": "manual_dense",
            "modelo": f"mlp_in{INPUT_WINDOW}_out{OUTPUT_WINDOW}",
            "n_params": int(r["n_params"]) if not pd.isna(r["n_params"]) else np.nan,
            "mae_train": float(r["mae_train"]),
            "mae_val": float(r["mae_val"]),
            "mae_test": float(r["mae_test"]),
            "notas": "MLP denso de competicion",
        })

df_comp = pd.DataFrame(comparativa_rows).sort_values("mae_test")
df_comp.to_csv(RESULTS_DIR / "autokeras_resultados.csv", index=False)
print("Comparativa final (ordenada por mae_test):")
print(df_comp.to_string(index=False))
print(f"\nGuardado: {RESULTS_DIR / 'autokeras_resultados.csv'}")

In [ ]:
if len(df_comp) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    colors = {"autokeras": "tab:red", "keras_tuner": "tab:orange",
              "manual": "tab:blue", "manual_ensemble": "tab:purple", "manual_dense": "tab:gray"}
    bar_colors = [colors.get(m, "black") for m in df_comp["metodo"]]
    y = range(len(df_comp))
    ax.barh(y, df_comp["mae_test"].values, color=bar_colors)
    ax.set_yticks(list(y))
    ax.set_yticklabels(df_comp["modelo"], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("MAE test")
    ax.set_title(f"Comparativa NAS vs manual ({INPUT_WINDOW}, {OUTPUT_WINDOW})", fontweight="bold")
    for i, v in enumerate(df_comp["mae_test"].values):
        ax.text(v, i, f" {v:.6f}", va="center", fontsize=8)

    ax = axes[1]
    df_params = df_comp.dropna(subset=["n_params"])
    if len(df_params) > 0:
        ax.scatter(df_params["n_params"], df_params["mae_test"], s=120, c=[colors.get(m, "black") for m in df_params["metodo"]])
        for _, r in df_params.iterrows():
            ax.annotate(r["modelo"], (r["n_params"], r["mae_test"]), fontsize=8, xytext=(5, 5), textcoords="offset points")
        ax.set_xscale("log")
        ax.set_xlabel("n_params (log)")
        ax.set_ylabel("MAE test")
        ax.set_title("Trade-off complejidad vs MAE")

    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "autokeras_comparativa.png", bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"guardado: {RESULTS_DIR / 'autokeras_comparativa.png'}")

## 5. Conclusión

Decisión automática basada en los resultados. La narrativa para la oral es la misma en ambos casos: **demuestra rigor metodológico** y posicionamiento crítico frente a herramientas automáticas.

In [ ]:
conclusion_lines = []

if len(df_comp) > 0 and ak_mae_test is not None:
    nas_row = df_comp[df_comp["metodo"].isin(["autokeras", "keras_tuner"])].iloc[0]
    manual_rows = df_comp[df_comp["metodo"].isin(["manual", "manual_ensemble", "manual_dense"])]
    if len(manual_rows) > 0:
        best_manual = manual_rows.sort_values("mae_test").iloc[0]
        delta = (best_manual["mae_test"] - nas_row["mae_test"]) / best_manual["mae_test"] * 100
        conclusion_lines.append(
            f"NAS ({nas_row['metodo']}) mae_test={nas_row['mae_test']:.6f} con {nas_row['n_params']:,} params."
        )
        conclusion_lines.append(
            f"Mejor manual: {best_manual['modelo']} mae_test={best_manual['mae_test']:.6f} con "
            f"{int(best_manual['n_params']) if not pd.isna(best_manual['n_params']) else '?'} params."
        )
        if delta > 0.5:
            conclusion_lines.append(
                f"-> NAS GANA por {delta:+.2f}%. La busqueda automatica encontro una arquitectura mejor que el diseno manual. "
                "Para la oral: 'reconocemos el valor de NAS cuando el espacio de hiperparametros es grande y el budget computacional permite explorarlo'."
            )
        elif delta < -0.5:
            conclusion_lines.append(
                f"-> Manual GANA por {-delta:+.2f}%. El conocimiento de dominio bate a la busqueda automatica con el presupuesto dado. "
                "Para la oral: 'el conocimiento financiero condensado en el diseno manual (variantes inspiradas en el profesor + Lopez de Prado) "
                "supera a NAS, validando la metodologia del curso'."
            )
        else:
            conclusion_lines.append(
                f"-> Empate tecnico ({delta:+.2f}%). Las dos opciones convergen a calidad similar. "
                "Para la oral: 'NAS y diseno manual llegan a soluciones equivalentes en este problema, sugiriendo que el limite de mejora viene "
                "dado por el SNR del problema (ruido financiero), no por la arquitectura'."
            )

    if ak_mae_train < ak_mae_val * 0.7:
        conclusion_lines.append(
            "Gap train/val notable en NAS: posible sobreajuste. NAS tiende a encontrar arquitecturas muy expresivas que sobreajustan el train."
        )

if not conclusion_lines:
    conclusion_lines.append("Sin resultados de NAS comparables. Instala autokeras o keras-tuner y vuelve a ejecutar.")

print("Conclusiones para la oral:")
print("=" * 80)
for i, line in enumerate(conclusion_lines, 1):
    print(f"{i}. {line}")

### Reflexión metodológica

- **AutoKeras** explora un espacio mucho mayor (puede mezclar `DenseBlock`, `RNNBlock`, `ConvBlock`, attention…). Es la opción más ambiciosa pero también la menos reproducible.
- **Keras Tuner** restringe a un espacio que **tú defines**. Mucho más reproducible, alineado con la metodología del Workshop 1 (donde se prioriza la transparencia del método).
- En problemas con **alto ruido** (cómo es el caso del forecasting de retornos diarios), **NAS rara vez bate al diseño cuidadoso** porque el límite de mejora viene dado por el SNR del problema, no por la arquitectura. Esto es **literatura conocida** (López de Prado, Marcos. *Advances in Financial Machine Learning*, cap. 1).
- El profesor (a quien le interesa enseñarte a diseñar bien, no a delegar) **probablemente valore más una conclusión honesta y razonada** que un "AutoKeras encontró el mejor modelo". Cualquiera de los dos resultados sirve para la oral.

### Para la presentación

Una sola transparencia con:
- Tabla `df_comp` (4-5 filas).
- Bar chart `autokeras_comparativa.png`.
- Frase de cierre: *"AutoKeras [bate / iguala / no bate] al diseño manual, lo cual [confirma / matiza / contradice] la hipótesis de que el SNR del problema es el cuello de botella en problemas financieros."*

## 6. Validación final + cheat sheet

In [ ]:
checks_total = 0
checks_ok = 0


def _do(label, cond, detail=""):
    global checks_total, checks_ok
    checks_total += 1
    icono = "[OK]" if cond else "[FAIL]"
    print(f"{icono} {label} {detail}")
    if cond:
        checks_ok += 1


print("Validacion del notebook 09")
print("=" * 80)

_do("AutoKeras O Keras Tuner disponible", USE_AUTOKERAS or USE_KERAS_TUNER)
_do("CSV autokeras_resultados.csv existe", (RESULTS_DIR / "autokeras_resultados.csv").exists())
if (RESULTS_DIR / "autokeras_resultados.csv").exists():
    df_check = pd.read_csv(RESULTS_DIR / "autokeras_resultados.csv")
    _do("CSV >= 2 filas (NAS + al menos 1 referencia)", len(df_check) >= 2, f"(actual: {len(df_check)})")

_do("Figura comparativa existe", (RESULTS_DIR / "autokeras_comparativa.png").exists())
if best_model is not None:
    _do("Modelo NAS guardado", (MODELS_DIR / "autokeras_in90_out90.keras").exists())

print("=" * 80)
print(f"Resultado: {checks_ok}/{checks_total} checks superados.")

### Cheat sheet

**Subir el presupuesto de búsqueda**: cambia `MAX_TRIALS = 30` y `EPOCHS_PER_TRIAL = 80` en la celda 3 (búsqueda) y vuelve a ejecutar.

**Forzar AutoKeras (sin fallback)**:

```python
import autokeras as ak  # falla aqui si no esta instalado
USE_AUTOKERAS = True
USE_KERAS_TUNER = False
```

**Cargar el modelo NAS guardado**:

```python
from tensorflow.keras.models import load_model
nas = load_model(MODELS_DIR / "autokeras_in90_out90.keras", compile=False)
```

**Reutilizar el modelo NAS en `08_carteras_2025.ipynb`**:

```python
nas = load_model(MODELS_DIR / "autokeras_in90_out90.keras", compile=False)

def predict_returns_nas(window):
    if window.ndim == 2:
        window = window[np.newaxis, ...]
    X_n = scaler_X_inference.transform(window.reshape(-1, window.shape[-1])).reshape(window.shape).astype(np.float32)
    pred = nas.predict(X_n, verbose=0)
    if pred.ndim == 3:
        pred = pred.reshape(pred.shape[0], -1)
    return scaler_y_inference.inverse_transform(pred)[0]

PRED_MODELS["nas"] = predict_returns_nas
```

### Cierre del proyecto

Este es el último de los 3 notebooks de tu parte. Resumen del entregable completo:

| Notebook | Rol | Salida principal |
|---|---|---|
| `06_mixtos.ipynb` | Competición | 80 modelos + matriz competición + deep dive |
| `07_investigacion.ipynb` | Investigación | 12 técnicas comparadas, mejor preprocesado por celda |
| `08_carteras_2025.ipynb` | Aplicación práctica | 6 carteras backtested en 2025 |
| `09_autokeras_extra.ipynb` | Extra defensa | NAS vs manual, reflexión metodológica |

Siguiente paso: redactar PDF de presentación (5 min de oral, 5 min de Q&A).